In [1]:
# CROSS VALIDATION FOR REGRESSION - FULL DEMO


import numpy as np

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# 1. LOAD DATASET

data = fetch_california_housing()
X = data.data
y = data.target

print("\nDataset Info")
print("Shape:", X.shape)
print("\n" + "="*50)


# 2. TRAIN-TEST SPLIT (BASELINE)

print("\n1. TRAIN-TEST SPLIT (BASELINE)")


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

baseline_r2 = model.score(X_test, y_test)

print("R2 Score (Baseline):", round(baseline_r2, 4))
print("\n" + "="*50)


# 3. INSTABILITY DEMO
print("\n2. INSTABILITY OF TRAIN-TEST SPLIT")


for i in range(5):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    model.fit(X_train, y_train)
    r2 = model.score(X_test, y_test)
    print(f"Run {i+1}: R2 = {round(r2, 4)}")

print("\n" + "="*50)


# 4. K-FOLD CROSS VALIDATION (R2)

print("\n3. K-FOLD CROSS VALIDATION (R2)")


kf = KFold(n_splits=5, shuffle=True, random_state=42)

r2_scores = cross_val_score(model, X, y, cv=kf, scoring='r2')

print("R2 Scores:", np.round(r2_scores, 4))
print("Mean R2:", round(r2_scores.mean(), 4))

print("\n" + "="*50)

# 5. CROSS VALIDATION WITH MSE & RMSE

print("\n4. CROSS VALIDATION (MSE & RMSE)")


mse_scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error')

rmse_scores = np.sqrt(-mse_scores)

print("RMSE Scores:", np.round(rmse_scores, 4))
print("Mean RMSE:", round(rmse_scores.mean(), 4))

print("\n" + "="*50)

# 6. SINGLE MODEL ASSESSMENT

print("\n5. SINGLE MODEL EVALUATION")


print("Mean R2:", round(r2_scores.mean(), 4))
print("Std Dev:", round(r2_scores.std(), 4))

print("\n" + "="*50)


# 7. COMPARING TWO MODELS

print("\n6. MODEL COMPARISON")


model1 = LinearRegression()
model2 = DecisionTreeRegressor()

r2_1 = cross_val_score(model1, X, y, cv=kf, scoring='r2')
r2_2 = cross_val_score(model2, X, y, cv=kf, scoring='r2')

print("Linear Regression Mean R2:", round(r2_1.mean(), 4))
print("Decision Tree Mean R2:", round(r2_2.mean(), 4))

print("\n" + "="*50)


# 8. PAIRED K-FOLD COMPARISON

print("\n7. PAIRED K-FOLD COMPARISON")
print("----------------------------")

scores1 = []
scores2 = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model1.fit(X_train, y_train)
    model2.fit(X_train, y_train)

    scores1.append(model1.score(X_test, y_test))
    scores2.append(model2.score(X_test, y_test))

print("Linear Regression Scores:", np.round(scores1, 4))
print("Decision Tree Scores:", np.round(scores2, 4))

diff = np.array(scores1) - np.array(scores2)
print("Difference:", np.round(diff, 4))

print("\n" + "="*50)


# 9. MODEL DRIFT SIMULATION

print("\n8. MODEL DRIFT SIMULATION")


model.fit(X_train, y_train)

# Simulate new data with noise
X_new = X_test + np.random.normal(0, 2, X_test.shape)

print("Before Drift R2:", round(model.score(X_test, y_test), 4))
print("After Drift R2:", round(model.score(X_new, y_test), 4))

print("\n" + "="*50)


# 10. FINAL SUMMARY

print("\n9. FINAL SUMMARY")


print("""
- Train-Test Split is unstable
- Cross Validation gives reliable performance
- R2 measures explained variance (higher is better)
- RMSE measures error (lower is better)
- Linear vs Tree shows model differences
- Paired CV ensures fair comparison
- Drift shows models degrade over time
""")


Dataset Info
Shape: (20640, 8)


1. TRAIN-TEST SPLIT (BASELINE)
R2 Score (Baseline): 0.5758


2. INSTABILITY OF TRAIN-TEST SPLIT
Run 1: R2 = 0.6251
Run 2: R2 = 0.5915
Run 3: R2 = 0.6101
Run 4: R2 = 0.6107
Run 5: R2 = 0.6179


3. K-FOLD CROSS VALIDATION (R2)
R2 Scores: [0.5758 0.6137 0.6086 0.6213 0.5875]
Mean R2: 0.6014


4. CROSS VALIDATION (MSE & RMSE)
RMSE Scores: [0.7456 0.7264 0.7136 0.7105 0.7451]
Mean RMSE: 0.7283


5. SINGLE MODEL EVALUATION
Mean R2: 0.6014
Std Dev: 0.017


6. MODEL COMPARISON
Linear Regression Mean R2: 0.6014
Decision Tree Mean R2: 0.616


7. PAIRED K-FOLD COMPARISON
----------------------------
Linear Regression Scores: [0.5758 0.6137 0.6086 0.6213 0.5875]
Decision Tree Scores: [0.6183 0.6173 0.6022 0.6276 0.6074]
Difference: [-0.0425 -0.0035  0.0063 -0.0063 -0.0199]


8. MODEL DRIFT SIMULATION
Before Drift R2: 0.5875
After Drift R2: -2.4881


9. FINAL SUMMARY

- Train-Test Split is unstable
- Cross Validation gives reliable performance
- R2 measures explain